#Lab9_File_Handling.ipynb.

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import csv
import io
import json
import html as html_lib

# ---------- Widgets ----------
file_uploader = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
student_dropdown = widgets.Dropdown(options=[], description='Student:', disabled=True,
                                    layout=widgets.Layout(width='400px'))
update_btn = widgets.Button(description='Update Report', button_style='success', icon='check')
pdf_btn = widgets.Button(description='Save to PDF', button_style='info', icon='file-pdf-o')
report_output = widgets.Output()

# ---------- State ----------
df = None                 # all subjects of all students
students = {}             # block_id -> {'name', 'course', 'year'}
current_report_html = ""


def error_box(msg):
    return HTML(f"<div style='color: #c62828; font-weight: bold; font-family: Arial; padding: 10px; "
                f"border: 1px solid #ffcdd2; background-color: #ffebee; border-radius: 5px;'>{msg}</div>")


def get_uploaded_bytes():
    """Works with both ipywidgets 7 (dict) and 8 (tuple of dicts)."""
    val = file_uploader.value
    if not val:
        return None
    item = list(val.values())[0] if isinstance(val, dict) else val[0]
    return bytes(item['content'])


def lowest_grade_color(grade):
    """Color for the LOWEST GRADE card (Philippine scale: higher number = worse)."""
    if grade <= 1.75:
        return '#e6a100'   # yellow / gold
    elif grade <= 3.00:
        return '#ef6c00'   # orange
    else:
        return '#c62828'   # red


def build_grade_table(sdf):
    """Builds the grade table by hand so alignment and widths are fully controlled."""
    esc = html_lib.escape
    th_base = ("background-color:#3b5c3b; color:white; padding:10px 12px; font-family:Arial,sans-serif; "
               "font-size:0.9em; letter-spacing:0.5px; -webkit-print-color-adjust:exact; print-color-adjust:exact;")
    td_base = "padding:10px 12px; border-bottom:1px solid #e2e8e2; color:#444; font-family:Arial,sans-serif; font-size:0.95em; vertical-align:middle;"

    # (header, column key, width, alignment)
    columns = [
        ('CODE',        'CODE',        '14%', 'left'),
        ('DISCRIPTION', 'DISCRIPTION', '38%', 'left'),
        ('UNITS',       'UNITS',       '8%',  'center'),
        ('TEACHER',     'TEACHER',     '28%', 'left'),
        ('GRADE',       'GRADE',       '12%', 'center'),
    ]

    html = ["<table style='width:100%; border-collapse:collapse; table-layout:fixed; margin-bottom:20px;'>"]
    html.append("<colgroup>" + "".join(f"<col style='width:{w}'>" for _, _, w, _ in columns) + "</colgroup>")
    html.append("<thead><tr>")
    for header, _, _, align in columns:
        html.append(f"<th style='{th_base} text-align:{align};'>{header}</th>")
    html.append("</tr></thead><tbody>")

    for i, (_, row) in enumerate(sdf.iterrows()):
        bg = '#f7faf7' if i % 2 else '#ffffff'
        html.append(f"<tr style='background-color:{bg}; -webkit-print-color-adjust:exact; print-color-adjust:exact;'>")
        for _, key, _, align in columns:
            value = str(row[key]).strip()
            if key == 'GRADE':
                num = pd.to_numeric(value, errors='coerce')
                if pd.notna(num):
                    value = f"{num:.2f}"
            html.append(f"<td style='{td_base} text-align:{align}; word-wrap:break-word;'>{esc(value)}</td>")
        html.append("</tr>")

    html.append("</tbody></table>")
    return "".join(html)


def parse_blocks(text):
    """
    Parses a CSV laid out like this, repeated for every student:

        name, course, year            <- optional label line
        Juan Dela Cruz, BSIT, 1st Year
        code, discription, units, teacher, grade   <- optional label line
        IT101, Intro to Computing, 3, Mr. Reyes, 1.75
        ...

    - A row with 3 filled cells   -> new student (name, course, year)
    - A row with 4-5 filled cells -> a subject of the current student
    - Blank rows and the label rows ("name..." / "code...") are skipped.
    """
    rows, studs = [], {}
    current = None
    block_id = 0

    for raw in csv.reader(io.StringIO(text)):
        cells = [c.strip().lstrip('\u2022').strip() for c in raw]
        while cells and cells[-1] == '':
            cells.pop()
        if not cells or all(c == '' for c in cells):
            continue

        first = cells[0].lower()
        if first == 'name' or first == 'code':
            continue

        if len(cells) == 3:
            block_id += 1
            current = block_id
            studs[current] = {'name': cells[0], 'course': cells[1], 'year': cells[2]}
        elif len(cells) in (4, 5) and current is not None:
            cells += [''] * (5 - len(cells))
            rows.append([current] + cells)

    cols = ['BLOCK', 'CODE', 'DISCRIPTION', 'UNITS', 'TEACHER', 'GRADE']
    return pd.DataFrame(rows, columns=cols), studs


def load_csv(change=None):
    """Runs when a CSV is uploaded: parses it and fills the student dropdown."""
    global df, students
    content = get_uploaded_bytes()
    if content is None:
        return

    with report_output:
        clear_output(wait=True)
        try:
            text = content.decode('utf-8-sig', errors='replace')
            data, studs = parse_blocks(text)
        except Exception as e:
            display(error_box(f"Error reading uploaded file: {html_lib.escape(str(e))}"))
            return

        if not studs:
            display(error_box("No students found. Each student block must start with a line like "
                              "<code>name, course, year</code> values."))
            return

        df, students = data, studs

        names = [s['name'] for s in students.values()]
        options = []
        for bid, s in students.items():
            label = s['name']
            if names.count(s['name']) > 1:
                label += f" ({s['course']}, {s['year']})"
            options.append((label, bid))
        options.sort(key=lambda o: o[0].lower())

        student_dropdown.unobserve(on_student_change, names='value')
        student_dropdown.options = options
        student_dropdown.value = options[0][1]
        student_dropdown.disabled = False
        student_dropdown.observe(on_student_change, names='value')

    generate_report_dashboard()


def on_student_change(change):
    generate_report_dashboard()


def save_pdf(b):
    global current_report_html
    if not current_report_html:
        display(HTML("<script>alert('Please upload a CSV and select a student first before saving to PDF!');</script>"))
        return

    escaped_html = json.dumps(current_report_html)
    js_print = f"""
    <script>
    (function() {{
        var reportHtml = {escaped_html};
        var printWindow = window.open('', '_blank', 'height=600,width=800');
        printWindow.document.write('<html><head><title>Student Academic Performance Report</title>');
        printWindow.document.write('<style>');
        printWindow.document.write('body {{ font-family: Arial, sans-serif; margin: 30px; color: #333; }}');
        printWindow.document.write('.report-title-header {{ background-color: #2e522e !important; color: white !important; padding: 15px !important; border-radius: 6px; text-align: center; font-weight: bold; -webkit-print-color-adjust: exact; print-color-adjust: exact; }}');
        printWindow.document.write('</style></head><body>');
        printWindow.document.write(reportHtml);
        printWindow.document.write('</body></html>');
        printWindow.document.close();
        printWindow.focus();
        setTimeout(function() {{
            printWindow.print();
            printWindow.close();
        }}, 500);
    }})();
    </script>
    """
    display(HTML(js_print))


def generate_report_dashboard(b=None):
    global current_report_html
    with report_output:
        clear_output(wait=True)

        if df is None:
            display(error_box("Please upload a student grades CSV file first."))
            return
        if student_dropdown.value is None:
            display(error_box("Please select a student."))
            return

        bid = student_dropdown.value
        info = students[bid]
        sdf = df[df['BLOCK'] == bid].copy()

        sdf['GRADE_NUM'] = pd.to_numeric(sdf['GRADE'], errors='coerce')
        sdf['UNITS_NUM'] = pd.to_numeric(sdf['UNITS'], errors='coerce')

        # Summary metrics (Philippine scale: lower number = better grade)
        graded = sdf.dropna(subset=['GRADE_NUM'])
        total_subjects = len(graded)
        total_units = sdf['UNITS_NUM'].sum()
        valid_units = graded['UNITS_NUM'].sum()
        weighted_gwa = (graded['GRADE_NUM'] * graded['UNITS_NUM']).sum() / valid_units if valid_units > 0 else 0
        highest_grade = graded['GRADE_NUM'].min() if total_subjects else 0
        lowest_grade = graded['GRADE_NUM'].max() if total_subjects else 0
        lowest_color = lowest_grade_color(lowest_grade)

        display(widgets.HTML("""
        <style>
            .report-title-header { background-color: #2e522e !important; color: white !important; padding: 15px !important; border-radius: 6px; text-align: center; font-weight: bold; }
        </style>
        """))

        grade_table = build_grade_table(sdf)

        esc = html_lib.escape
        printable_content = f"""
        <div id="print-area">
            <div style="font-family: Arial, sans-serif; padding: 20px; border-radius: 8px; background-color: #f4f7f4; border: 1px solid #d0ded0; margin-bottom: 15px;">
                <h2 class="report-title-header">Student Academic Performance Report</h2>
                <div style="display: flex; justify-content: space-between; font-size: 1.4em; color: #4e6e4e; font-weight: 500; margin-top: 15px;">
                    <span><strong>Name:</strong> {esc(info['name'])}</span>
                    <span><strong>Course:</strong> {esc(info['course'])}</span>
                    <span><strong>Year:</strong> {esc(info['year'])}</span>
                </div>
            </div>

            <h3 style='color: #4e6e4e; margin-top: 10px; margin-bottom: 10px; font-family: Arial, sans-serif;'>Grade Report</h3>
            {grade_table}

            <div style="font-family: Arial, sans-serif; padding: 15px; border-radius: 8px; background-color: #f0f4f0; border: 1px solid #cfe2cf; display: flex; justify-content: space-around; text-align: center; margin-top: 20px;">
                <div>
                    <div style="font-size: 0.9em; color: #5c7a5c; font-weight: bold;">TOTAL SUBJECTS</div>
                    <div style="font-size: 1.5em; color: #4e6e4e; font-weight: bold;">{total_subjects}</div>
                </div>
                <div style="border-left: 1px solid #cfe2cf; height: 40px;"></div>
                <div>
                    <div style="font-size: 0.9em; color: #5c7a5c; font-weight: bold;">UNITS EARNED</div>
                    <div style="font-size: 1.5em; color: #4e6e4e; font-weight: bold;">{total_units:.1f}</div>
                </div>
                <div style="border-left: 1px solid #cfe2cf; height: 40px;"></div>
                <div>
                    <div style="font-size: 0.9em; color: #5c7a5c; font-weight: bold;">BEST GRADE</div>
                    <div style="font-size: 1.5em; color: #2e7d32; font-weight: bold;">{highest_grade:.2f}</div>
                </div>
                <div style="border-left: 1px solid #cfe2cf; height: 40px;"></div>
                <div>
                    <div style="font-size: 0.9em; color: #5c7a5c; font-weight: bold;">LOWEST GRADE</div>
                    <div style="font-size: 1.5em; color: {lowest_color}; font-weight: bold;">{lowest_grade:.2f}</div>
                </div>
                <div style="border-left: 1px solid #cfe2cf; height: 40px;"></div>
                <div>
                    <div style="font-size: 0.9em; color: #5c7a5c; font-weight: bold;">AVERAGE GRADE (GWA)</div>
                    <div style="font-size: 1.5em; color: #4e6e4e; font-weight: bold;">{weighted_gwa:.2f}</div>
                </div>
            </div>
        </div>
        """
        current_report_html = printable_content
        display(widgets.HTML(printable_content))


# ---------- Events ----------
file_uploader.observe(load_csv, names='value')
student_dropdown.observe(on_student_change, names='value')
update_btn.on_click(generate_report_dashboard)
pdf_btn.on_click(save_pdf)

update_btn.style.button_color = '#7fa97f'
pdf_btn.style.button_color = '#6ba3a3'

# ---------- Layout ----------
gui_container = widgets.VBox([
    widgets.HTML("<h3 style='color: #4e6e4e; border-bottom: 2px solid #a3bfa3; padding-bottom: 5px; margin-bottom: 15px;'>Dashboard Settings & File Upload</h3>"),
    widgets.HBox([file_uploader, student_dropdown]),
    widgets.HBox([update_btn, pdf_btn], layout=widgets.Layout(margin='10px 0 0 0')),
    widgets.HTML("<hr style='border: 1px solid #cfe2cf; margin: 20px 0;'/>"),
    report_output
], layout=widgets.Layout(border='2px solid #a3bfa3', padding='20px', border_radius='12px',
                         background_color='#fafbfa', margin='10px 0'))

display(gui_container)

with report_output:
    display(HTML("""
    <div style="font-family: Arial, sans-serif; text-align: center; color: #5c7a5c; padding: 30px; border: 1px dashed #a3bfa3; border-radius: 8px; background-color: #fcfdfc;">
        <h3 style="margin-top: 0; color: #4e6e4e;">Welcome to the Grade Report Generator!</h3>
        <p><strong>Upload a CSV File</strong> above, then choose a <strong>Student</strong> from the dropdown to view their grades.</p>
    </div>
    """))

#file handling pastel theme

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import csv
import io
import json
import html as html_lib

# ---------- Pixel theme ----------
PIX = "'Press Start 2P','Courier New',monospace"   # headings / buttons
BODY = "'VT323','Courier New',monospace"            # readable pixel text for tables

THEME_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Press+Start+2P&family=VT323&display=swap');

/* ---- notched pixel panel ---- */
.px-panel {
    background: #fff0f3 !important;
    box-shadow: 0 -4px 0 0 #f0b6c6, 0 4px 0 0 #f0b6c6, -4px 0 0 0 #f0b6c6, 4px 0 0 0 #f0b6c6 !important;
}

/* ---- pixel buttons ---- */
.px-btn {
    font-family: 'Press Start 2P','Courier New',monospace !important;
    font-size: 11px !important;
    letter-spacing: 1px;
    text-transform: uppercase;
    color: #ffffff !important;
    background: var(--fill) !important;
    border: none !important;
    border-radius: 0 !important;
    margin: 8px !important;
    text-shadow: 2px 2px 0 var(--edge);
    box-shadow:
        0 -3px 0 0 var(--edge), 0 3px 0 0 var(--edge),
        -3px 0 0 0 var(--edge), 3px 0 0 0 var(--edge),
        inset 0 3px 0 0 var(--hi), inset 0 -3px 0 0 var(--lo) !important;
    cursor: pointer;
}
.px-btn:hover  { filter: brightness(1.06); }
.px-btn:active { transform: translateY(3px); }
.px-btn i, .px-btn .fa { display: none; }

.px-pink   { --fill:#f4c2cf; --edge:#e59bb0; --hi:#f9d9e2; --lo:#eaa9bb; }
.px-green  { --fill:#bfe6cf; --edge:#8fcaa8; --hi:#d6f1e2; --lo:#a3d8ba; }
.px-blue   { --fill:#c3dbf2; --edge:#86b3de; --hi:#dbeaf8; --lo:#a2c6e8; }
.px-purple { --fill:#d6c0ea; --edge:#b497d4; --hi:#e8d9f5; --lo:#c4a8e0; }
.px-yellow { --fill:#f8e58c; --edge:#e3c445; --hi:#fcf0b5; --lo:#efd566; }

/* ---- pixel dropdown ---- */
.px-select select {
    font-family: 'VT323','Courier New',monospace !important;
    font-size: 20px !important;
    height: 38px !important;
    color: #6b4f8a !important;
    background: #f0e4f8 !important;
    border: 3px solid #b497d4 !important;
    border-radius: 0 !important;
    padding: 0 8px !important;
}
.px-select select:disabled { opacity: .6; }
</style>
"""

# ---------- Widgets ----------
file_uploader = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV',
                                   layout=widgets.Layout(width='190px', height='42px'))
file_uploader.add_class('px-btn'); file_uploader.add_class('px-blue')

student_dropdown = widgets.Dropdown(options=[], description='', disabled=True,
                                    layout=widgets.Layout(width='330px', margin='6px 0 0 0'))
student_dropdown.add_class('px-select')

update_btn = widgets.Button(description='Update', layout=widgets.Layout(width='150px', height='42px'))
update_btn.add_class('px-btn'); update_btn.add_class('px-green')

pdf_btn = widgets.Button(description='Save PDF', layout=widgets.Layout(width='150px', height='42px'))
pdf_btn.add_class('px-btn'); pdf_btn.add_class('px-pink')

report_output = widgets.Output()

# ---------- State ----------
df = None
students = {}
current_report_html = ""


def error_box(msg):
    return HTML(f"<div style=\"font-family:{BODY}; font-size:22px; color:#b0405a; padding:12px; "
                f"border:3px solid #e59bb0; background:#ffe4ea;\">{msg}</div>")


def get_uploaded_bytes():
    """Works with both ipywidgets 7 (dict) and 8 (tuple of dicts)."""
    val = file_uploader.value
    if not val:
        return None
    item = list(val.values())[0] if isinstance(val, dict) else val[0]
    return bytes(item['content'])


def lowest_grade_color(grade):
    """Color for the LOWEST GRADE card (Philippine scale: higher number = worse)."""
    if grade <= 1.75:
        return '#e0a000'   # yellow / gold
    elif grade <= 3.00:
        return '#ef6c00'   # orange
    else:
        return '#d32f2f'   # red


def build_grade_table(sdf):
    esc = html_lib.escape
    th = (f"background-color:#b497d4; color:#ffffff; padding:12px; font-family:{PIX}; font-size:10px; "
          f"letter-spacing:1px; text-shadow:2px 2px 0 #8e6fb5; border-bottom:4px solid #9c7cc0;")
    td = (f"padding:8px 12px; border-bottom:2px solid #f3dfe8; color:#5b4a6b; font-family:{BODY}; "
          f"font-size:21px; vertical-align:middle; word-wrap:break-word;")

    columns = [
        ('CODE',        'CODE',        '14%', 'left'),
        ('DISCRIPTION', 'DISCRIPTION', '38%', 'left'),
        ('UNITS',       'UNITS',       '8%',  'center'),
        ('TEACHER',     'TEACHER',     '28%', 'left'),
        ('GRADE',       'GRADE',       '12%', 'center'),
    ]

    out = ["<table style='width:100%; border-collapse:collapse; table-layout:fixed; margin-bottom:20px; "
           "border:4px solid #b497d4;'>"]
    out.append("<colgroup>" + "".join(f"<col style='width:{w}'>" for _, _, w, _ in columns) + "</colgroup>")
    out.append("<thead><tr>")
    for header, _, _, align in columns:
        out.append(f"<th style='{th} text-align:{align};'>{header}</th>")
    out.append("</tr></thead><tbody>")

    for i, (_, row) in enumerate(sdf.iterrows()):
        bg = '#fdf4f8' if i % 2 else '#ffffff'
        out.append(f"<tr style='background-color:{bg};'>")
        for _, key, _, align in columns:
            value = str(row[key]).strip()
            if key == 'GRADE':
                num = pd.to_numeric(value, errors='coerce')
                if pd.notna(num):
                    value = f"{num:.2f}"
            out.append(f"<td style='{td} text-align:{align};'>{esc(value)}</td>")
        out.append("</tr>")

    out.append("</tbody></table>")
    return "".join(out)


def parse_blocks(text):
    """
    name, course, year                          <- optional label line
    Juan Dela Cruz, BSIT, 1st Year              <- 3 values = new student
    code, discription, units, teacher, grade    <- optional label line
    IT101, Intro to Computing, 3, Mr. Reyes, 1.75   <- 4-5 values = subject
    """
    rows, studs = [], {}
    current = None
    block_id = 0

    for raw in csv.reader(io.StringIO(text)):
        cells = [c.strip().lstrip('\u2022').strip() for c in raw]
        while cells and cells[-1] == '':
            cells.pop()
        if not cells or all(c == '' for c in cells):
            continue

        first = cells[0].lower()
        if first == 'name' or first == 'code':
            continue

        if len(cells) == 3:
            block_id += 1
            current = block_id
            studs[current] = {'name': cells[0], 'course': cells[1], 'year': cells[2]}
        elif len(cells) in (4, 5) and current is not None:
            cells += [''] * (5 - len(cells))
            rows.append([current] + cells)

    cols = ['BLOCK', 'CODE', 'DISCRIPTION', 'UNITS', 'TEACHER', 'GRADE']
    return pd.DataFrame(rows, columns=cols), studs


def load_csv(change=None):
    global df, students
    content = get_uploaded_bytes()
    if content is None:
        return

    with report_output:
        clear_output(wait=True)
        try:
            text = content.decode('utf-8-sig', errors='replace')
            data, studs = parse_blocks(text)
        except Exception as e:
            display(error_box(f"Error reading uploaded file: {html_lib.escape(str(e))}"))
            return

        if not studs:
            display(error_box("No students found. Each student block must start with a line like "
                              "<code>name, course, year</code> values."))
            return

        df, students = data, studs

        names = [s['name'] for s in students.values()]
        options = []
        for bid, s in students.items():
            label = s['name']
            if names.count(s['name']) > 1:
                label += f" ({s['course']}, {s['year']})"
            options.append((label, bid))
        options.sort(key=lambda o: o[0].lower())

        student_dropdown.unobserve(on_student_change, names='value')
        student_dropdown.options = options
        student_dropdown.value = options[0][1]
        student_dropdown.disabled = False
        student_dropdown.observe(on_student_change, names='value')

    generate_report_dashboard()


def on_student_change(change):
    generate_report_dashboard()


def save_pdf(b):
    global current_report_html
    if not current_report_html:
        display(HTML("<script>alert('Please upload a CSV and select a student first before saving to PDF!');</script>"))
        return

    escaped_html = json.dumps(current_report_html)
    js_print = f"""
    <script>
    (function() {{
        var reportHtml = {escaped_html};
        var printWindow = window.open('', '_blank', 'height=600,width=800');
        printWindow.document.write('<html><head><title>Student Academic Performance Report</title>');
        printWindow.document.write('<link href="https://fonts.googleapis.com/css2?family=Press+Start+2P&family=VT323&display=swap" rel="stylesheet">');
        printWindow.document.write('<style>body {{ margin: 30px; -webkit-print-color-adjust: exact; print-color-adjust: exact; }} * {{ -webkit-print-color-adjust: exact; print-color-adjust: exact; }}</style>');
        printWindow.document.write('</head><body>');
        printWindow.document.write(reportHtml);
        printWindow.document.write('</body></html>');
        printWindow.document.close();
        printWindow.focus();
        setTimeout(function() {{
            printWindow.print();
            printWindow.close();
        }}, 1000);
    }})();
    </script>
    """
    display(HTML(js_print))


def stat_card(label, value, value_color, border, label_color):
    return (f"<div style='flex:1; margin:0 6px; padding:14px 6px; background:#ffffff; border:4px solid {border}; "
            f"text-align:center;'>"
            f"<div style=\"font-family:{PIX}; font-size:9px; line-height:1.5; color:{label_color};\">{label}</div>"
            f"<div style=\"font-family:{BODY}; font-size:40px; color:{value_color}; margin-top:6px;\">{value}</div>"
            f"</div>")


def generate_report_dashboard(b=None):
    global current_report_html
    with report_output:
        clear_output(wait=True)

        if df is None:
            display(error_box("Please upload a student grades CSV file first."))
            return
        if student_dropdown.value is None:
            display(error_box("Please select a student."))
            return

        bid = student_dropdown.value
        info = students[bid]
        sdf = df[df['BLOCK'] == bid].copy()

        sdf['GRADE_NUM'] = pd.to_numeric(sdf['GRADE'], errors='coerce')
        sdf['UNITS_NUM'] = pd.to_numeric(sdf['UNITS'], errors='coerce')

        graded = sdf.dropna(subset=['GRADE_NUM'])
        total_subjects = len(graded)
        total_units = sdf['UNITS_NUM'].sum()
        valid_units = graded['UNITS_NUM'].sum()
        weighted_gwa = (graded['GRADE_NUM'] * graded['UNITS_NUM']).sum() / valid_units if valid_units > 0 else 0
        highest_grade = graded['GRADE_NUM'].min() if total_subjects else 0
        lowest_grade = graded['GRADE_NUM'].max() if total_subjects else 0
        lowest_color = lowest_grade_color(lowest_grade)

        grade_table = build_grade_table(sdf)
        esc = html_lib.escape

        chip = (f"display:inline-block; padding:8px 14px; font-family:{BODY}; font-size:24px; color:#5b4a6b; "
                f"border:3px solid {{b}}; background:{{bg}};")

        cards = "".join([
            stat_card("TOTAL SUBJECTS", total_subjects, '#d9728f', '#f0b6c6', '#d9728f'),
            stat_card("UNITS EARNED", f"{total_units:.1f}", '#4f9d73', '#a9dcc0', '#4f9d73'),
            stat_card("BEST GRADE", f"{highest_grade:.2f}", '#4a85c0', '#a9cbe9', '#4a85c0'),
            stat_card("LOWEST GRADE", f"{lowest_grade:.2f}", lowest_color, '#ead27a', '#b8962a'),
            stat_card("AVERAGE (GWA)", f"{weighted_gwa:.2f}", '#8a62b5', '#c9b0e2', '#8a62b5'),
        ])

        printable_content = f"""
        <div id="print-area" style="padding:6px;">
            <div style="padding:18px; background:#fff0f3; border:4px solid #f0b6c6; margin-bottom:16px;">
                <div style="background:#ef9db5; border:4px solid #e08aa3; padding:16px; text-align:center;
                            font-family:{PIX}; font-size:15px; color:#ffffff; text-shadow:3px 3px 0 #d0728d;
                            letter-spacing:2px;">
                    STUDENT ACADEMIC PERFORMANCE REPORT
                </div>
                <div style="display:flex; justify-content:space-between; flex-wrap:wrap; gap:10px; margin-top:16px;">
                    <span style="{chip.format(b='#a9cbe9', bg='#e3effa')}"><b>Name:</b> {esc(info['name'])}</span>
                    <span style="{chip.format(b='#a9dcc0', bg='#e4f6ec')}"><b>Course:</b> {esc(info['course'])}</span>
                    <span style="{chip.format(b='#ead27a', bg='#fdf5cc')}"><b>Year:</b> {esc(info['year'])}</span>
                </div>
            </div>

            <div style="font-family:{PIX}; font-size:12px; color:#9c7cc0; margin:10px 0 12px 0;">GRADE REPORT</div>
            {grade_table}

            <div style="display:flex; margin:20px -6px 0 -6px;">{cards}</div>
        </div>
        """
        current_report_html = printable_content
        display(widgets.HTML(printable_content))


# ---------- Events ----------
file_uploader.observe(load_csv, names='value')
student_dropdown.observe(on_student_change, names='value')
update_btn.on_click(generate_report_dashboard)
pdf_btn.on_click(save_pdf)

# ---------- Layout ----------
title_banner = widgets.HTML(
    f"<div style=\"display:inline-block; padding:12px 20px; background:#f8e58c; color:#ffffff; "
    f"font-family:{PIX}; font-size:13px; letter-spacing:3px; text-shadow:2px 2px 0 #e3c445; "
    f"box-shadow:0 -3px 0 0 #e3c445, 0 3px 0 0 #e3c445, -3px 0 0 0 #e3c445, 3px 0 0 0 #e3c445, "
    f"inset 0 3px 0 0 #fcf0b5, inset 0 -3px 0 0 #efd566; margin:6px;\">DASHBOARD SETTINGS</div>"
)
student_label = widgets.HTML(
    f"<div style=\"font-family:{PIX}; font-size:10px; color:#8a62b5; padding:16px 8px 0 14px;\">STUDENT</div>"
)

gui_container = widgets.VBox([
    widgets.HTML(THEME_CSS),
    title_banner,
    widgets.HBox([file_uploader, student_label, student_dropdown],
                 layout=widgets.Layout(margin='14px 0 0 0', align_items='flex-start')),
    widgets.HBox([update_btn, pdf_btn], layout=widgets.Layout(margin='4px 0 0 0')),
    widgets.HTML("<div style='height:4px; background:#f0b6c6; margin:18px 0;'></div>"),
    report_output
], layout=widgets.Layout(padding='22px', margin='14px 10px', border='none'))
gui_container.add_class('px-panel')

display(gui_container)

with report_output:
    display(HTML(f"""
    <div style="text-align:center; padding:30px; border:4px dashed #c9b0e2; background:#fbf6ff;">
        <div style="font-family:{PIX}; font-size:13px; color:#8a62b5; line-height:1.8;">WELCOME TO THE<br>GRADE REPORT GENERATOR!</div>
        <div style="font-family:{BODY}; font-size:24px; color:#7a6a8a; margin-top:14px;">
            Press <b style="color:#4a85c0;">UPLOAD CSV</b>, then pick a <b style="color:#8a62b5;">STUDENT</b> to see their grades.
        </div>
    </div>
    """))